# TP 6 — Texte, Transformers et LLM avec Hugging Face

**M1 · Deep Learning & Big Data · Séance 6 · Environnement : Google Colab (GPU)**

| Partie | Contenu |
|---|---|
| 1 | Utiliser un modèle Transformer en une ligne : les pipelines |
| 2 | Regarder sous le capot : la tokenisation |
| 3 | Les embeddings : mesurer le sens des phrases |
| 4 | Classer sans entraîner : le *zero-shot* |
| 5 | Passer à l'échelle : prédire sur des milliers de critiques (Pandas puis Spark) |
| 6 | Bonus : un petit LLM qui génère du texte |

Complétez les `____` des cellules `# TODO` et répondez aux questions ❓.

In [ ]:
!pip install -q transformers sentence-transformers datasets pyspark
import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from transformers import pipeline

DEVICE = 0 if torch.cuda.is_available() else -1
print("GPU :", torch.cuda.get_device_name(0) if DEVICE == 0 else "aucun (ce sera plus lent)")

---
## Partie 1 — Les pipelines

Un `pipeline` Hugging Face enchaîne trois étapes : **tokeniser** le texte, le passer dans le **modèle**, puis **interpréter** la sortie. On choisit une tâche et un modèle du Hub.

On commence par un modèle d'avis multilingue qui prédit une note de 1 à 5 étoiles.

In [ ]:
notation = pipeline("sentiment-analysis",
                    model="nlptown/bert-base-multilingual-uncased-sentiment",
                    device=DEVICE)

avis = [
    "Ce cours est vraiment passionnant, j'ai tout compris !",
    "Le TP était trop long et les consignes pas claires.",
    "Bof, ni bien ni mal.",
]
for texte, r in zip(avis, notation(avis)):
    print(f"{r['label']:<8} ({r['score']:.0%})  ←  {texte}")

### 🛠️ Exercice 1.1

Écrivez trois avis de votre choix, dont un **ironique** (« Génial, encore une panne… ») et un avec une **négation** (« pas mal du tout »). Le modèle s'en sort-il mieux que le sac de mots du TP 4 ?

In [ ]:
mes_avis = [
    ____,   # TODO
    ____,   # TODO
    ____,   # TODO
]
for texte, r in zip(mes_avis, notation(mes_avis)):
    print(f"{r['label']:<8} ({r['score']:.0%})  ←  {texte}")

---
## Partie 2 — La tokenisation

Avant d'entrer dans le modèle, le texte est découpé en **tokens** : des mots, des morceaux de mots ou des signes de ponctuation. Chaque token a un numéro dans le vocabulaire du modèle.

In [ ]:
from transformers import AutoTokenizer

tok_fr = AutoTokenizer.from_pretrained("almanach/camembert-base")       # modèle entraîné sur du français
tok_en = AutoTokenizer.from_pretrained("google-bert/bert-base-uncased")  # modèle entraîné sur de l'anglais

phrase = "Les étudiants entraînent un réseau de neurones anticonstitutionnellement."
for nom, tok in [("CamemBERT (fr)", tok_fr), ("BERT (en)", tok_en)]:
    tokens = tok.tokenize(phrase)
    print(f"{nom:<15} {len(tokens):>2} tokens : {tokens}")

In [ ]:
# Les numéros que le modèle reçoit vraiment
encodage = tok_fr(phrase)
print("Identifiants :", encodage["input_ids"])
print("Retour au texte :", tok_fr.decode(encodage["input_ids"]))

### 🛠️ Exercice 2.1

Tokenisez avec les deux tokeniseurs une phrase en anglais de votre choix, puis un mot très rare ou inventé. Comptez les tokens.

In [ ]:
for texte in [____, ____]:          # TODO : une phrase anglaise, un mot rare ou inventé
    for nom, tok in [("CamemBERT", tok_fr), ("BERT", tok_en)]:
        print(f"{nom:<10} {tok.tokenize(texte)}")
    print()

### ❓ Question 2.2

Pourquoi un même texte ne donne-t-il pas le même nombre de tokens selon le modèle ? Pourquoi est-ce important quand on paie une API de LLM ?

*Votre réponse :*

---
## Partie 3 — Les embeddings

Un **embedding** est un vecteur de nombres qui représente le sens d'un texte. Deux phrases de sens proche ont des vecteurs proches, même sans mot en commun. On mesure la proximité par la **similarité cosinus** : 1 = même direction, 0 = sans rapport.

In [ ]:
from sentence_transformers import SentenceTransformer, util

encodeur = SentenceTransformer("sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2",
                               device="cuda" if DEVICE == 0 else "cpu")

phrases = [
    "Comment réinitialiser mon mot de passe ?",
    "J'ai oublié mes identifiants de connexion.",
    "Quels sont les horaires de la bibliothèque ?",
    "À quelle heure ferme la BU ?",
    "Le chat dort sur le canapé.",
]
vecteurs = encodeur.encode(phrases)
print("Chaque phrase devient un vecteur de", vecteurs.shape[1], "nombres")

sim = util.cos_sim(vecteurs, vecteurs).numpy()
plt.figure(figsize=(7, 6))
plt.imshow(sim, cmap="Blues", vmin=0, vmax=1)
plt.xticks(range(len(phrases)), [p[:22] + "…" for p in phrases], rotation=45, ha="right")
plt.yticks(range(len(phrases)), [p[:22] + "…" for p in phrases])
for i in range(len(phrases)):
    for j in range(len(phrases)):
        plt.text(j, i, f"{sim[i, j]:.2f}", ha="center", va="center", fontsize=8)
plt.title("Similarité entre phrases"); plt.colorbar(); plt.tight_layout(); plt.show()

### 🛠️ Exercice 3.1 — Un moteur de recherche sémantique

On a une petite FAQ. Pour une question posée, trouvez l'entrée de la FAQ **la plus proche en sens**.

In [ ]:
faq = [
    "Pour changer votre mot de passe, allez dans Mon compte puis Sécurité.",
    "La bibliothèque est ouverte de 8 h à 20 h du lundi au vendredi.",
    "Les notes sont publiées sur l'ENT deux semaines après l'examen.",
    "Le Wi-Fi étudiant s'appelle eduroam ; connectez-vous avec vos identifiants de l'école.",
    "Pour toute absence, envoyez un justificatif au secrétariat sous 48 h.",
]
vecteurs_faq = encodeur.encode(faq)

question = "je n'arrive pas à me connecter à internet sur le campus"
v_question = encodeur.encode(____)                    # TODO : encoder la question
scores = util.cos_sim(v_question, ____)[0].numpy()     # TODO : comparer aux vecteurs de la FAQ

for i in scores.argsort()[::-1][:3]:
    print(f"{scores[i]:.2f}  {faq[i]}")

### ❓ Question 3.2

La question ne contient ni « Wi-Fi » ni « eduroam ». Pourquoi la bonne réponse est-elle trouvée ? Citez une application de cette technique.

*Votre réponse :*

---
## Partie 4 — Classer sans entraîner : le zero-shot

Un modèle *zero-shot* classe un texte dans des catégories **qu'on invente au moment de l'appel**, sans aucun exemple d'entraînement.

In [ ]:
zero_shot = pipeline("zero-shot-classification",
                     model="MoritzLaurer/mDeBERTa-v3-base-mnli-xnli",
                     device=DEVICE)

message = "Bonjour, ma carte étudiante ne fonctionne plus à la cantine depuis lundi."
categories = ["problème technique", "question pédagogique", "restauration", "logement"]

r = zero_shot(message, candidate_labels=categories)
for etiquette, score in zip(r["labels"], r["scores"]):
    print(f"{score:6.1%}  {etiquette}")

### 🛠️ Exercice 4.1

Inventez vos propres catégories (par exemple pour trier les e-mails d'un service client, ou les articles d'un journal) et testez trois messages.

In [ ]:
mes_categories = [____]          # TODO : 3 à 5 catégories
mes_messages = [____, ____, ____]  # TODO : 3 messages

for m in mes_messages:
    r = zero_shot(m, candidate_labels=mes_categories)
    print(f"{r['labels'][0]:<25} ({r['scores'][0]:.0%})  ←  {m}")

---
## Partie 5 — Passer à l'échelle

On applique un modèle de sentiment anglais aux critiques IMDb préparées au TP 3, puis on compare au réseau du TP 4.

### 5.1 Charger les critiques de test

In [ ]:
import os
candidats = ["imdb_propre", "/content/drive/MyDrive/DeepLearning/imdb_propre"]
if not any(os.path.isdir(c) for c in candidats):
    try:
        from google.colab import drive
        drive.mount("/content/drive")
    except Exception:
        pass
chemin = next((c for c in candidats if os.path.isdir(c)), None)

if chemin:
    test_df = pd.read_parquet(os.path.join(chemin, "split=test"))
else:
    from datasets import load_dataset
    test_df = load_dataset("stanfordnlp/imdb", split="test").to_pandas().rename(columns={"text": "texte_propre"})
    test_df["texte_propre"] = test_df["texte_propre"].str.replace(r"<[^>]+>", " ", regex=True).str.lower()

echantillon = test_df.sample(1000, random_state=0).reset_index(drop=True)
print(len(echantillon), "critiques à classer")

### 🛠️ Exercice 5.2 — Prédire par lots avec Pandas

Le paramètre `batch_size` envoie les critiques au GPU par paquets ; `truncation=True` coupe les critiques trop longues pour le modèle (512 tokens au maximum).

In [ ]:
import time
sentiment_en = pipeline("sentiment-analysis",
                        model="distilbert/distilbert-base-uncased-finetuned-sst-2-english",
                        device=DEVICE)

debut = time.time()
resultats = sentiment_en(echantillon["texte_propre"].tolist(), batch_size=____, truncation=____)   # TODO
duree = time.time() - debut

echantillon["prediction"] = [1 if r["label"] == "POSITIVE" else 0 for r in resultats]
precision = (echantillon["prediction"] == echantillon["label"]).mean()
print(f"{len(echantillon)} critiques en {duree:.1f} s · bonnes réponses : {precision:.1%}")

### ❓ Question 5.3

Comparez avec le réseau sac de mots du TP 4. Ce modèle n'a **jamais vu IMDb** pendant son entraînement (il a été affiné sur un autre corpus de critiques, SST-2). Qu'en concluez-vous ?

*Votre réponse :*

### 5.4 Le même traitement distribué avec Spark

Sur des millions de critiques, une seule machine ne suffit plus. Avec Spark, on applique le modèle **en parallèle sur chaque partition** grâce à une `pandas_udf` : Spark envoie à chaque tâche un paquet de lignes sous forme de série Pandas, la fonction renvoie les prédictions.

Ici tout tourne sur la même machine Colab (et sur CPU dans les tâches Spark) : on garde 400 critiques pour que ce soit rapide. Le code serait **identique** sur un cluster.

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.functions import pandas_udf

spark = SparkSession.builder.appName("TP6").config("spark.driver.memory", "4g").getOrCreate()

critiques = spark.createDataFrame(echantillon[["texte_propre", "label"]].head(400)).repartition(4)
print("Partitions :", critiques.rdd.getNumPartitions())

MODELE_SENTIMENT = "distilbert/distilbert-base-uncased-finetuned-sst-2-english"
_modele = None   # chargé à la demande dans chaque tâche Spark, puis réutilisé

@pandas_udf("int")
def predire(textes: pd.Series) -> pd.Series:
    global _modele
    if _modele is None:
        from transformers import pipeline
        _modele = pipeline("sentiment-analysis", model=MODELE_SENTIMENT, device=-1)
    sorties = _modele(textes.tolist(), batch_size=16, truncation=True)
    return pd.Series([1 if s["label"] == "POSITIVE" else 0 for s in sorties])

### 🛠️ Exercice 5.5

Ajoutez la colonne `prediction` en appliquant la fonction `predire` à la colonne de texte, puis calculez la proportion de bonnes réponses.

In [ ]:
avec_pred = critiques.withColumn("prediction", predire(F.col(____)))     # TODO

(avec_pred
 .agg(F.avg((F.col("prediction") == F.col(____)).cast("int")).alias("bonnes_reponses"))   # TODO
 .show())

In [ ]:
# Les résultats s'écrivent comme n'importe quel DataFrame
avec_pred.write.mode("overwrite").parquet("imdb_predictions")
spark.stop()

### ❓ Question 5.6

Pourquoi charge-t-on le modèle **dans** la fonction (à la première utilisation) plutôt qu'une fois au début du notebook ?

*Votre réponse :*

---
## Partie 6 — Bonus : un petit LLM qui génère du texte

Un LLM (*Large Language Model*) est un Transformer entraîné à **prédire le token suivant**, puis affiné pour suivre des instructions. On essaie un très petit modèle ouvert (0,5 milliard de paramètres) : il tient sur le GPU gratuit de Colab, mais il est loin des grands assistants.

In [ ]:
generateur = pipeline("text-generation", model="Qwen/Qwen2.5-0.5B-Instruct", device=DEVICE)

messages = [{"role": "user", "content": "Explique en trois phrases simples ce qu'est le Big Data."}]
sortie = generateur(messages, max_new_tokens=150, do_sample=False)
print(sortie[0]["generated_text"][-1]["content"])

### ❓ Question 6.1

Posez-lui une question factuelle précise (une date, un chiffre, une personne peu connue). La réponse est-elle exacte ? Comment appelle-t-on ce phénomène et comment le limiter ?

*Votre réponse :*

---
## 🎯 Pour vérifier que vous avez compris

1. Qu'est-ce qu'un token ?
2. Que représente un embedding, et comment compare-t-on deux embeddings ?
3. Pourquoi utiliser Spark pour appliquer un modèle à des millions de textes ?

*Vos réponses :*